# LC 62 — Unique Paths
**Day 65 | 2D Dynamic Programming | Medium**

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> Every cell can only be reached from
above or from the left. So the number of unique paths to any cell
is exactly the sum of paths to those two neighbors. Fill the grid
once — bottom-right answer appears automatically.
</div>

## Official Problem Statement

There is a robot on an `m x n` grid. The robot is initially located
at the **top-left corner** (i.e., `grid[0][0]`). The robot tries to
move to the **bottom-right corner** (i.e., `grid[m-1][n-1]`). The
robot can only move either **down** or **right** at any point.

Given the two integers `m` and `n`, return the number of possible
unique paths that the robot can take to reach the bottom-right corner.

**Constraints:**
- `1 <= m, n <= 100`
- The answer will be less than or equal to `2 * 10^9`.

## What This Is Actually Asking

You have a grid. A robot starts top-left and must reach bottom-right
using only down or right moves. How many distinct routes exist?

The robot must make exactly `(m-1)` down moves and `(n-1)` right
moves in some order — the answer is the number of ways to arrange
those moves. You can compute it combinatorially, but DP is more
intuitive and generalizes to obstacle variants.

Each cell stores: "how many ways can I be reached?" Build up from
the top-left corner to get the final answer at the bottom-right.

## Walk Through an Example by Hand

**Input:** `m=3, n=3`

Step 1 — Initialize: first row and first column all = 1,
because there is only one way to reach any cell in row 0
(go right) or col 0 (go down).

```
dp start:
[1, 1, 1]
[1, ?, ?]
[1, ?, ?]
```

Step 2 — Fill row 1:
- dp[1][1] = dp[0][1] + dp[1][0] = 1 + 1 = 2
- dp[1][2] = dp[0][2] + dp[1][1] = 1 + 2 = 3

Step 3 — Fill row 2:
- dp[2][1] = dp[1][1] + dp[2][0] = 2 + 1 = 3
- dp[2][2] = dp[1][2] + dp[2][1] = 3 + 3 = **6**

**Answer: 6**

## The Picture

```
  3x3 grid — dp table values

  col:  0    1    2
row 0 [ 1 ][ 1 ][ 1 ]
row 1 [ 1 ][ 2 ][ 3 ]
row 2 [ 1 ][ 3 ][*6*]  <-- answer

  Rule: dp[i][j] = dp[i-1][j] + dp[i][j-1]
  Base: dp[0][j] = 1  (top row)
        dp[i][0] = 1  (left col)

  Every cell = paths from top + paths from left
```

1D space optimization: keep a single row array and update
left-to-right. `dp[j] += dp[j-1]` replaces the 2D fill.

## When To Use This Pattern

- When a problem asks "how many ways" to traverse a grid from
  corner to corner — think 2D DP path counting.
- When movement is restricted (only right/down) — think each
  cell = sum of valid predecessors.
- When you see a grid with a single start and single end point
  — think DP table filled from top-left to bottom-right.
- When a 2D DP table uses only the previous row — think 1D
  rolling array to reduce space to O(n).
- When the problem adds obstacles — think same pattern with
  `dp[i][j] = 0` at blocked cells.

## The Approach

Create a 1D DP array of length `n`, initialized to all 1s
(representing the first row). For each subsequent row, update
left-to-right: `dp[j] += dp[j-1]`, because `dp[j]` still holds
the value from the row above, and `dp[j-1]` is the updated
value from the left in the current row. After processing all
rows, `dp[n-1]` is the answer.

In [ ]:
from typing import List

In [ ]:
def test_harness(func):
    """Run tests for unique_paths."""
    tests = [
        # (m, n, expected)
        (3, 7, 28),
        (3, 2, 3),
        (3, 3, 6),
        (1, 1, 1),   # edge: single cell
        (1, 5, 1),   # edge: single row
        (5, 1, 1),   # edge: single col
        (2, 2, 2),
        (10, 10, 48620),
    ]
    passed = 0
    for m, n, expected in tests:
        result = func(m, n)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        else:
            print(
                f"  {status}: m={m} n={n} "
                f"expected={expected} got={result}"
            )
    total = len(tests)
    print(f"\nResults: {passed}/{total} passed")
    if passed == total:
        print("All tests PASSED!")

In [ ]:
def unique_paths(m: int, n: int) -> int:
    """
    Count unique paths from top-left to bottom-right
    in an m x n grid, moving only right or down.

    Approach: 1D DP rolling array.
    - dp[j] represents paths to current row, column j.
    - Initialize all 1s (first row has exactly one path).
    - For each new row, update left-to-right:
        dp[j] += dp[j-1]
      dp[j] (old) = paths from above row, same col.
      dp[j-1] (new) = paths from left in current row.

    Args:
        m: number of rows
        n: number of columns

    Returns:
        Number of unique paths (int)

    Examples:
        >>> unique_paths(3, 7)
        28
        >>> unique_paths(1, 1)
        1
    """
    # Debug: show inputs
    print(f"[DEBUG] m={m}, n={n}")
    pass

In [ ]:
# Uncomment and run when solution is ready
# test_harness(unique_paths)

## Complexity

| Approach | Time | Space | Notes |
|---|---|---|---|
| Brute Force (recursion) | O(2^(m+n)) | O(m+n) | Re-computes subproblems |
| 2D DP table | O(m*n) | O(m*n) | Intuitive, easy to follow |
| 1D rolling array | O(m*n) | O(n) | Optimal; reuse single row |
| Combinatorics | O(min(m,n)) | O(1) | C(m+n-2, m-1) directly |

## Real World Connection

At **Citi**, trade settlement workflows route transactions through
a pipeline of stages — counting valid routing paths through a
compliance and approval grid is exactly this problem. On **AWS**,
a data pipeline DAG from ingestion to serving has many valid
execution orderings; unique path counting helps estimate the
fan-out. In **Data Engineering**, partitioned ETL jobs sometimes
need to enumerate all valid data flows through a processing grid
to schedule resources. The rolling-array optimization matters
when the grid represents a massive state space, keeping memory
usage linear rather than quadratic.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra